In [ ]:
#%pip install scipy
#%pip install statsmodels

In [4]:
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
from scipy.stats import spearmanr, pearsonr
import glob

In [6]:

# === CONFIGURAZIONE ===
# percorso della cartella con i CSV puliti
path = "G:/Il mio Drive/Desktop/Magistrale/IoT_ESP_SleepSense/Misurazioni/*.csv"
# === CARICAMENTO FILES ===
all_files = glob.glob(path)
dfs = []

for file in all_files:
    df = pd.read_csv(file)
    dfs.append(df)

# unisci tutti i file in un unico dataframe
df_all = pd.concat(dfs, ignore_index=True)

# === PULIZIA E TIPI ===
df_all["_time"] = pd.to_datetime(df_all["_time"])
cols_to_numeric = ["humidity", "light", "mic", "temperature", "is_moving"]
df_all[cols_to_numeric] = df_all[cols_to_numeric].apply(pd.to_numeric, errors="coerce")

# rimuovi eventuali righe con valori nulli
df_all = df_all.dropna(subset=cols_to_numeric)

print("Dimensione dataset:", df_all.shape)

Dimensione dataset: (633, 10)


In [ ]:
# === ANALISI CORRELAZIONE ===
# matrice di correlazione Pearson
corr_matrix = df_all[cols_to_numeric].corr(method="pearson")
print("\nMatrice di correlazione Pearson:\n", corr_matrix)

# calcolo anche Spearman tra movimento e altri sensori
for col in ["humidity", "light", "mic", "temperature"]:
    spear_corr, _ = spearmanr(df_all["is_moving"], df_all[col])
    pear_corr, _ = pearsonr(df_all["is_moving"], df_all[col])
    print(f"\nCorrelazione con {col}:")
    print(f"  Pearson:  {pear_corr:.3f}")
    print(f"  Spearman: {spear_corr:.3f}")

# === GRAFICI ===
# 1. Line chart di tutte le variabili
fig = px.line(df_all, x="_time", y=cols_to_numeric, title="Sensor Data Over Time")
fig.update_layout(xaxis_title="Time", yaxis_title="Sensor Values")
fig.show()

# 2. Heatmap delle correlazioni
fig_corr = ff.create_annotated_heatmap(
    z=corr_matrix.values,
    x=list(corr_matrix.columns),
    y=list(corr_matrix.index),
    annotation_text=corr_matrix.round(2).values,
    showscale=True,
    colorscale="RdBu",
    reversescale=True
)
fig_corr.update_layout(title="Correlation Heatmap (Pearson)")
fig_corr.show()

# 3. Scatter plot movimento vs ogni variabile
for col in ["humidity", "light", "mic", "temperature"]:
    fig_scatter = px.scatter(df_all, x=col, y="is_moving", trendline="ols",
                             title=f"Movimento vs {col}")
    fig_scatter.show()



Matrice di correlazione Pearson:
              humidity     light       mic  temperature  is_moving
humidity     1.000000 -0.224267 -0.354406    -0.554055  -0.119223
light       -0.224267  1.000000 -0.014042     0.112788   0.002747
mic         -0.354406 -0.014042  1.000000     0.374961   0.324786
temperature -0.554055  0.112788  0.374961     1.000000  -0.062837
is_moving   -0.119223  0.002747  0.324786    -0.062837   1.000000

Correlazione con humidity:
  Pearson:  -0.119
  Spearman: -0.087

Correlazione con light:
  Pearson:  0.003
  Spearman: 0.102

Correlazione con mic:
  Pearson:  0.325
  Spearman: 0.097

Correlazione con temperature:
  Pearson:  -0.063
  Spearman: 0.033
